# 🌱 Crop Recommendation System

**B.Tech CSE AI/ML Project – Google Colab Notebook**

This notebook loads a crop recommendation dataset, performs EDA, trains multiple machine-learning classification models, compares their performance, and predicts a suitable crop from soil and climate conditions.


## 1. Import Libraries

In [ ]:
import io
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

print("Libraries imported successfully!")


## 2. Load the Dataset

The notebook uses the same public crop-recommendation dataset used by the Streamlit application.

Required columns: `n`, `p`, `k`, `temperature`, `humidity`, `ph`, `rainfall`, and `label`.


In [ ]:
DATASET_URL = "https://raw.githubusercontent.com/Gladiator07/Harvestify/master/Data-processed/crop_recommendation.csv"

with urllib.request.urlopen(DATASET_URL, timeout=30) as response:
    raw_df = pd.read_csv(io.BytesIO(response.read()))

print("Dataset loaded successfully.")
print("Shape:", raw_df.shape)
display(raw_df.head())


## 3. Optional: Upload Your Own CSV

In [ ]:
# Run this cell only if you want to replace the downloaded dataset with your own CSV.

from google.colab import files

uploaded = files.upload()

if uploaded:
    filename = next(iter(uploaded))
    raw_df = pd.read_csv(io.BytesIO(uploaded[filename]))
    print("Uploaded:", filename)
    print("Shape:", raw_df.shape)
    display(raw_df.head())


## 4. Clean and Prepare the Dataset

In [ ]:
FEATURES = ["n", "p", "k", "temperature", "humidity", "ph", "rainfall"]

df = raw_df.copy()
df.columns = [str(c).strip().lower() for c in df.columns]

if "label" in df.columns:
    TARGET = "label"
elif "crop" in df.columns:
    TARGET = "crop"
else:
    raise ValueError("Dataset must contain a 'label' or 'crop' column.")

missing = [c for c in FEATURES if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[FEATURES + [TARGET]].copy()

for col in FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[TARGET] = df[TARGET].astype(str).str.strip()
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=FEATURES + [TARGET])
df = df[df[TARGET] != ""].drop_duplicates().reset_index(drop=True)

print("Cleaned dataset shape:", df.shape)
print("Number of crop classes:", df[TARGET].nunique())
display(df.head())


## 5. Dataset Information

In [ ]:
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing Values"))

print("\nDescriptive statistics:")
display(df[FEATURES].describe().round(2))


## 6. Exploratory Data Analysis (EDA)

### Crop Class Distribution

In [ ]:
plt.figure(figsize=(14, 6))
sns.countplot(data=df, x=TARGET, order=sorted(df[TARGET].unique()))
plt.xticks(rotation=90)
plt.title("Crop Class Distribution")
plt.xlabel("Crop")
plt.ylabel("Number of Samples")
plt.tight_layout()
plt.show()


### Feature Distributions

In [ ]:
df[FEATURES].hist(figsize=(14, 10), bins=20)
plt.suptitle("Feature Distributions", y=1.02)
plt.tight_layout()
plt.show()


### Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df[FEATURES].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()


## 7. Train-Test Split

In [ ]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 8. Create and Train Machine Learning Models

Models used:
1. Decision Tree
2. Random Forest
3. Gaussian Naive Bayes
4. Support Vector Machine (SVM)


In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        n_jobs=-1
    ),
    "Naive Bayes": GaussianNB(),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(kernel="rbf"))
    ])
}

trained_models = {}
training_results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model

    train_pred = model.predict(X_train)
    training_results.append({
        "Model": name,
        "Training Accuracy (%)": round(
            accuracy_score(y_train, train_pred) * 100, 2
        )
    })

training_table = pd.DataFrame(training_results)
display(training_table)


## 9. Test Accuracy Comparison

In [ ]:
test_results = []

for name, model in trained_models.items():
    predictions = model.predict(X_test)
    test_results.append({
        "Model": name,
        "Test Accuracy (%)": round(
            accuracy_score(y_test, predictions) * 100, 2
        )
    })

test_table = pd.DataFrame(test_results).sort_values(
    "Test Accuracy (%)", ascending=False
).reset_index(drop=True)

display(test_table)


In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=test_table, x="Model", y="Test Accuracy (%)")
plt.ylim(0, 100)
plt.xticks(rotation=20)
plt.title("Test Accuracy Comparison")
plt.tight_layout()
plt.show()


## 10. Stratified Cross-Validation

In [ ]:
folds = min(5, int(y_train.value_counts().min()))

if folds >= 2:
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    cv_results = []

    for name, model in models.items():
        scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring="accuracy"
        )

        cv_results.append({
            "Model": name,
            "Mean CV Accuracy (%)": round(scores.mean() * 100, 2),
            "CV Std Dev (%)": round(scores.std() * 100, 2)
        })

    cv_table = pd.DataFrame(cv_results).sort_values(
        "Mean CV Accuracy (%)", ascending=False
    )
    display(cv_table)
else:
    print("Not enough samples per class for cross-validation.")


## 11. Detailed Evaluation

In [ ]:
best_model_name = test_table.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_predictions = best_model.predict(X_test)

print("Selected model:", best_model_name)
print("\nClassification Report:")
print(classification_report(y_test, best_predictions, zero_division=0))


In [ ]:
labels = sorted(y.unique())
matrix = confusion_matrix(y_test, best_predictions, labels=labels)

plt.figure(figsize=(13, 10))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    xticklabels=labels,
    yticklabels=labels,
    cmap="Blues"
)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted Crop")
plt.ylabel("Actual Crop")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 12. Feature Importance

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importance = pd.Series(
        best_model.feature_importances_,
        index=FEATURES
    ).sort_values(ascending=False)

    display(importance.to_frame("Importance").round(4))

    plt.figure(figsize=(9, 5))
    sns.barplot(x=importance.values, y=importance.index)
    plt.title(f"Feature Importance - {best_model_name}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()
else:
    print("Built-in feature importance is not available for the selected model.")


## 13. Crop Prediction

Change the values below to test the trained model with new soil and climate conditions.


In [ ]:
input_data = pd.DataFrame([{
    "n": 90,
    "p": 42,
    "k": 43,
    "temperature": 25.0,
    "humidity": 80.0,
    "ph": 6.5,
    "rainfall": 200.0
}])

predicted_crop = best_model.predict(input_data)[0]

print("Input values:")
display(input_data)
print("Recommended Crop:", str(predicted_crop).upper())


## 14. Conclusion

The project demonstrates how machine-learning classification algorithms can recommend crops from soil nutrients and environmental conditions. Multiple models are trained and evaluated using test accuracy and stratified cross-validation, and the selected model is used for prediction.

**Note:** This is a machine-learning prediction and should be combined with local agricultural guidance before making real planting decisions.
